# Titanic Dataset ML Project
Dataset from Kaggle [Titanic Challenge](https://www.kaggle.com/competitions/titanic/data)

## Setup

### Import Packages

In [ ]:
# Include tqdm for monitoring progress as things run
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import sklearn as sk
import pandas as pd
import numpy as np
import xgboost
import logging
import optuna
import re

# For tqdm output to be the only cell output during hyperparam optimization.
optuna.logging.get_logger("optuna").setLevel(logging.WARNING)

## Dataset

### Import and Data Cleaning Function:

In [ ]:
def xy_dataset(filepath):
    '''
    Takes in dataset filepath 
    Returns x_train, and y_train, (shuffled and ready to be fitted)
    '''
    ## Importing data with pandas, dropping 
    DATA = pd.read_csv(filepath) # prev: ('data/train.csv')4
    y_train = DATA['Survived'] # To be predicted
    # Remove columns that can (should) not be predictors
    x_train = DATA.drop(['Survived'], axis=1)

    ## Clean dataset check, for debugging only
    def dataset_check(x_tr):
        '''To check for the number of NA/empty values in any column'''
        check_empties = []
        for column in x_tr.columns:
            tmp_list = [column, int(np.sum(x_tr[column].isna()))]
            #print(column)
            #print(np.sum(x_tr[column].isna()))
            check_empties.append(tmp_list)
        print(check_empties)

    # Uncomment below to check which columns have missing/NA values
    #dataset_check(x_train)

    ## Impute missing age values as median of Name title (Mr. Mrs. Ms. ...), Pclass (1, 2, 3), and Sex (Male, Female)
    def impute_age(x_tr):
        '''
        Imputing missing age as median of: Name's title/honorific,
        Pclass (social standing), and Sex.
        '''
        ## Find all unique titles/honorifics:
        # Assume:   Title starts after ", " and ends after ".", 
        #           also assume no missing names in dataset
        # Strategy: Strip all names down using regex and lstrip/rstrip
        Titles = []
        for name in x_tr['Name']: # For all names in the dataset
            match = re.search(", .*?\\. ",name).group() # The part of the string that matches our conditions to start at ', ' and end at '. '
            Titles.append(match.lstrip(', ').rstrip(' ')) # Append the honorific stripped of prefix/suffix
        UTitles = set(Titles) # All unique titles

        # To check, uncomment below
        #print(len(UTitles)," unique titles / honorifics")
        #print(UTitles)

        ## Imputing age values by sex, social status, then by title/honorific
        
        # For cases where these subdivisions do exist
        unique_titles = x_tr['Name'].str.extract(f"({'|'.join(UTitles)})", expand=False) # Searching for any titles in the unique list
        x_tr['Age'] = x_tr['Age'].fillna(x_tr.groupby(['Sex', 'Pclass', unique_titles])['Age'].transform('median')) 

        # If no other examples within sex/social status/title exist, impute those rows' age with global median
        global_median = x_tr['Age'].median()
        x_tr['Age'] = x_tr['Age'].fillna(global_median)

        return x_tr

    ## Drop irrelevant columns
    def drop_cols(x_tr):
        '''
        Drop irrelevant columns in input data matrix. (Placed in a function to alter as necessary)
        Placed after Age imputation as title / honorific extracted from inside Name.
        '''
        x_tr = x_tr.drop(columns=['Name','Cabin','PassengerId','Ticket']) # Passenger ID needs to be saved later for expected Kaggle submission
        return x_tr

    # Encode categrocial columns into dummy columns so numeric-centric models will take the data 
    def encoding_multi_categories(x_tr):
        '''
        Imputing missing Embarked entries with the mode.
        Encoding categorical data as numerical values and dummy columns so numeric models are able to function appropriately.
        '''
        # Encode sex with pd dummies

        # Imputing embarked categorical with most often occuring value, no missing values for sex (excluded)
        x_tr['Embarked'] = x_tr['Embarked'].fillna(x_tr['Embarked'].mode()[0]) 

        # Convert categoricals to OHE columns
        x_tr_enc = pd.get_dummies(x_tr, columns=['Embarked','Sex'], drop_first=True) # Try with changing drop_true to False, likely no change

        # Convert T/F to 1/0
        x_tr_enc = x_tr_enc.astype(float)

        return x_tr_enc


    ## Applying missing Age values imputation
    x_train = impute_age(x_train)

    ## Dropping irrelevant columns
    x_train = drop_cols(x_train)

    ## Encoding categorical columns to dummy columns with numerical values for True/False
    x_train = encoding_multi_categories(x_train)

    ## Shuffling training examples
    x_train, y_train = sk.utils.shuffle(x_train,y_train,random_state=12) #Change random state to None later

    return x_train, y_train
    

### Applying above function for training data

In [ ]:
x_train, y_train

### Applying function for test data 

In [ ]:
#x_train.dtypes

Pclass        int64
Name            str
Sex             str
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Cabin           str
Embarked        str
dtype: object

### Clean Dataset Check

[['Pclass', 0], ['Name', 0], ['Sex', 0], ['Age', 177], ['SibSp', 0], ['Parch', 0], ['Fare', 0], ['Cabin', 687], ['Embarked', 2]]


### Imputing Values Required?

#### Imputing Age

17  unique titles / honorifics
{'Jonkheer.', 'Master.', 'Major.', 'Sir.', 'Mr.', 'Mrs.', 'Rev.', 'Miss.', 'Col.', 'Mlle.', 'Lady.', 'Capt.', 'Mme.', 'Don.', 'Dr.', 'Ms.', 'the Countess.'}


#### Imputing Cabin (removing numeric vals, keeping alpha designation)

In [12]:
# Not implemented yet

### Encoding Required?

In [14]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [16]:
# Depends on the model used, not for RF/XGB
# Converting string columns to categories
# Use OHE

#x_train_encoded = pd.get_dummies(x_train,drop_first=True)

#for col in x_train.columns:
#    if (x_train[col].dtype == str) or (x_train[col].dtype == object):
#        print(col)
#        x_train[col] = x_train[col].astype('category')


### Shuffling Training Examples

In [19]:
x_train_encoded.head()

,Pclass,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Sex_male
456,1.0,65.0,0.0,0.0,26.5500,0.0,1.0,1.0
351,1.0,39.5,0.0,0.0,35.0000,0.0,1.0,1.0
173,3.0,21.0,0.0,0.0,7.9250,0.0,1.0,1.0
671,1.0,31.0,1.0,0.0,52.0000,0.0,1.0,1.0
836,3.0,21.0,0.0,0.0,8.6625,0.0,1.0,1.0


In [20]:
y_train.head() # checking if examples still ordered wrt x_train_shuff; verified

456    0
351    0
173    0
671    0
836    0
Name: Survived, dtype: int64

### Imputing Values if Required

## Model

### Model Setup

In [21]:
# Models used (Estimators dict / list)
# Use: Random Forest, XGBoost, LightGBM (Not the last one)
estimators = {
    #[name, model]
    'random_forest':   sk.ensemble.RandomForestClassifier,
    'xgboost':         xgboost.XGBClassifier 
    }

In [22]:
estimators.items()

dict_items([('random_forest', <class 'sklearn.ensemble._forest.RandomForestClassifier'>), ('xgboost', <class 'xgboost.sklearn.XGBClassifier'>)])

### Training

In [24]:
# model.fit(x_train,y_train)
models = {}

for model_name, model in estimators.items():
    #print(model_name)
    trained = model().fit(x_train_encoded,y_train)
    models[model_name] = trained
    trained = None
    #print(model)

### Initial Results

In [25]:
# Visualized; accuracy, PR Curves, AUC table summary
# model.predict(x_test)
# sklearn.metrics.score(y_train,y_test)

print("Where 0 represents an individual passenger's survival "
        "and 1 represents a single passenger not surviving:\n")

CLASS_REPORT_DICT = {}

for model_name in models:
    y_pred = models[model_name].predict(x_train_encoded)
    class_rep = sk.metrics.classification_report(y_train,y_pred)
    CLASS_REPORT_DICT[model_name] = sk.metrics.classification_report(y_train,y_pred,output_dict=True)

    print(f"Classification Report for {model_name.replace("_"," ").capitalize()} model:\n{class_rep}")
    #accuracy = sk.metrics.accuracy_score(y_train, y_pred)
    #AUC = sk.metrics.auc(y_train,y_pred)

    #
    # print("Accuracy of",model_name,"model: ",accuracy)
    #print("AUC of",model_name,"model: ",AUC)

Where 0 represents an individual passenger's survival and 1 represents a single passenger not surviving:

Classification Report for Random forest model:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       549
           1       0.99      0.97      0.98       342

    accuracy                           0.98       891
   macro avg       0.98      0.98      0.98       891
weighted avg       0.98      0.98      0.98       891

Classification Report for Xgboost model:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       549
           1       0.98      0.95      0.96       342

    accuracy                           0.97       891
   macro avg       0.97      0.97      0.97       891
weighted avg       0.97      0.97      0.97       891



## Hyperparameter Optimization

### Optuna Objective Function
Takes in an optuna trial object, sets up and trains ML model, returns evaluation metric

In [27]:
## Copied from above:
#estimators = {# [name, model]
#                    'random_forest':   sk.ensemble.RandomForestClassifier(),
#                    'xgboost':         xgboost.XGBClassifier() 
#                    }

In [51]:
def objective(trial):
    x, y = x_train_encoded, y_train
    
    model_name = trial.suggest_categorical("classifier", ["random_forest","xgboost"])

    if model_name == 'random_forest':
        params = {
            'n_estimators': trial.suggest_int(name="Number of Estimators",low=100,high=1000,step=100),
            'max_depth': trial.suggest_int(name="RF Maximum Depth",low=1,high=301,step=10),
            'max_features': trial.suggest_categorical("RF Max Features",["sqrt","log2"])
            }
    elif model_name == 'xgboost':
        params = {
            'learning_rate': trial.suggest_float(name="Learning Rate",low=0.1,high=0.9),
            'gamma': trial.suggest_float(name="Minimum Split Loss",low=0,high=0.8),
            'max_depth': trial.suggest_int(name="XGB Maximum Depth",low=2,high=18)
            }
    else: return AssertionError("Model used not in estimators list")

    clf = estimators[model_name](**params)

    # Accuracy across 3 folds
    score = sk.model_selection.cross_val_score(clf,x,y,cv=5).mean() 
    
    return score


### Trials

### Studies

In [52]:
N_TRIALS = 1000

# Using TQDM to visualize training progress
with tqdm(total=N_TRIALS,desc="Optimizing Models",unit="trial") as pbar:

    def tqdm_callback(study,trial):
        if study.best_trials:
            pbar.set_postfix({"Best Score": f"{study.best_value:.4f}"})
        pbar.update(1)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(n_startup_trials=10))
    study.optimize(objective,n_trials=N_TRIALS,callbacks=[tqdm_callback])

Optimizing Models:   0%|          | 0/1000 [00:00<?, ?trial/s]

In [54]:
print(f"Best model performance: {study.best_value}\n"
      f"Best model hyperparam configuration: {study.best_params}")

Best model performance: 0.8350260498399347
Best model hyperparam configuration: {'classifier': 'xgboost', 'Learning Rate': 0.4393638129229109, 'Minimum Split Loss': 0.3108796174731829, 'XGB Maximum Depth': 2}


### Train & Validation Sets

### Optimize Against Validation Set

### Perfomance Versus Unoptimized

### Results